# Kham pha va lam sach du lieu - BAF Base Dataset

Kiem tra chat luong du lieu, xu ly gia tri sentinel -1 va xuat du lieu chuan hoa sang CSV.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

sys.path.append("..")
from src.data.load_data import load_raw_dataset
from src.features.clean_and_engineer import clean_and_transform_pipeline


### Thong ke nhanh du lieu goc qua DuckDB


In [ ]:
conn = duckdb.connect()
conn.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(fraud_bool) AS fraud_count,
        ROUND(AVG(fraud_bool) * 100, 3) AS fraud_rate_pct,
        COUNT(DISTINCT month) AS num_months
    FROM "../data/raw/Base.csv"
""").df()

### Kiem tra ty le gia tri sentinel (-1)


In [ ]:
conn.execute("""
    SELECT 
        COUNT(*) AS total,
        ROUND(COUNT(CASE WHEN prev_address_months_count = -1 THEN 1 END) * 100.0 / COUNT(*), 2) AS prev_address_missing_pct,
        ROUND(COUNT(CASE WHEN intended_balcon_amount < 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS intended_balcon_missing_pct,
        ROUND(COUNT(CASE WHEN bank_months_count = -1 THEN 1 END) * 100.0 / COUNT(*), 2) AS bank_months_missing_pct,
        ROUND(COUNT(CASE WHEN session_length_in_minutes < 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS session_length_missing_pct
    FROM "../data/raw/Base.csv"
""").df()

### Chay pipeline lam sach va xuat CSV


In [ ]:
raw_df = load_raw_dataset("Base.csv")
clean_df = clean_and_transform_pipeline(raw_df)

out_path = Path("../data/processed/clean_base.csv")
clean_df.to_csv(out_path, index=False)
print(f"Đã lưu tệp kết quả tại {out_path} ({out_path.stat().st_size / (1024**2):.1f} MB)")